In [ ]:
!pip install -q transformers datasets


import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)
from datasets import load_dataset

#to know if the GPU Working or not
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


model_name = "gpt2"
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = GPT2LMHeadModel.from_pretrained(model_name).to(device)
model.config.pad_token_id = model.config.eos_token_id


dataset = load_dataset("wikitext", "wikitext-2-raw-v1")
dataset = dataset.filter(lambda ex: ex["text"].strip() != "")


def tokenize_fn(ex):
    return tokenizer(ex["text"], truncation=True, max_length=128)
tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])


block_size = 128
def group_texts(ex):
    ids = sum(ex["input_ids"], [])
    n = len(ids) // block_size
    chunks = [ids[i*block_size:(i+1)*block_size] for i in range(n)]
    return {
        "input_ids": chunks,
        "attention_mask": [[1]*block_size]*n
    }
lm_datasets = tokenized.map(group_texts, batched=True)


data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


training_args = TrainingArguments(
    output_dir="/content/gpt2-finetuned",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    eval_steps=500,
    save_steps=500,
    logging_steps=100,
    warmup_steps=200,
    save_total_limit=2,
    fp16=(device=="cuda"),
    do_train=True,
    do_eval=True,
    report_to=["none"],     
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
)


trainer.train()


eval_results = trainer.evaluate()
print("Perplexity:", torch.exp(torch.tensor(eval_results["eval_loss"])).item())


prompt = "The future of AI is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)
out = model.generate(
    **inputs,
    max_new_tokens=1,
    do_sample=False,
    pad_token_id=tokenizer.eos_token_id
)
print("Next word:", tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:]))
